# Milestone 1

In [12]:
import pandas as pd
df=pd.read_csv('../data/batch2_contracts_125rows.csv')
df.head()

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low


In [13]:
def upload_contract(path):
    return path

In [14]:
def extract_text_simulated(raw_text):
    return raw_text

In [15]:
import pdfplumber

def extract_text_pdf(path):
    text = ""
    with pdfplumber.open("../data/"+ path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

In [16]:
processed = []

for i, row in df.iterrows():
    text = extract_text_simulated(row['raw_text'])
    
    processed.append({
        "contract_id": row['contract_id'],
        "extracted_text": text
    })

df_extracted = pd.DataFrame(processed)
df_extracted.head()

,contract_id,extracted_text
0,1,This contract between Prime Auto and Emily Sto...
1,2,This contract between ABC Motors and Priya Sha...
2,3,This contract between DriveEasy Finance and Sa...
3,4,This contract between National Motors and Alex...
4,5,This contract between National Motors and John...


In [17]:
import re

def parse_fields(text):
    result = {}

    # APR
    apr = re.search(r'APR[: ]+(\d+\.?\d*)%', text)
    result['apr'] = float(apr.group(1)) if apr else None

    # Term
    term = re.search(r'(\d+)\s+months', text)
    result['term_months'] = int(term.group(1)) if term else None

    # Payment
    pay = re.search(r'payment[: ]+\$?(\d+)', text)
    result['monthly_payment'] = int(pay.group(1)) if pay else None

    # Penalty
    penalty = re.search(r'(late fee \$\d+|early termination fee \$\d+|none)', text.lower())
    result['penalty'] = penalty.group(1) if penalty else None

    return result

parsed = df_extracted['extracted_text'].apply(parse_fields)
parsed_df = pd.DataFrame(parsed.tolist())
parsed_df.head()

,apr,term_months,monthly_payment,penalty
0,10.49,24,959,none
1,3.78,24,804,early termination fee $300
2,5.41,24,774,late fee $50
3,5.26,48,1028,late fee $25
4,5.97,36,1180,none


In [18]:
output = pd.concat([df_extracted, parsed_df], axis=1)
output.to_csv("../data/milestone1_output_sathsreehari_k.csv", index=False)

In [19]:
df.info()
df.describe()
df.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   contract_id         125 non-null    int64  
 1   raw_text            125 non-null    object 
 2   apr                 125 non-null    float64
 3   term_months         125 non-null    int64  
 4   monthly_payment     125 non-null    int64  
 5   penalty_clause      93 non-null     object 
 6   recommended_action  125 non-null    object 
 7   risk_flag           125 non-null    object 
dtypes: float64(1), int64(3), object(4)
memory usage: 7.9+ KB


,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
23,24,This contract between Prime Auto and John Doe ...,6.40,48,648,Early termination fee $300,Approve,Low
105,106,This contract between National Motors and Rahu...,4.36,48,287,Late fee $50,Approve,High
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High
103,104,This contract between National Motors and John...,8.37,60,548,Late fee $50,Approve,Medium
84,85,This contract between National Motors and Emil...,4.40,36,684,Late fee $25,Reject,Low


In [20]:
# We will now build a simple parser that extracts 
# APR, term, payments, penalties using Python rules.
# This gives:

# APR

# Term

# Monthly payment

# Penalty

In [21]:
def parse_contract(text):
    result = {}

    # Extract APR
    import re
    apr_match = re.search(r'APR (\d+\.?\d*)%', text)
    result['apr_extracted'] = float(apr_match.group(1)) if apr_match else None

    # Extract term
    term_match = re.search(r'(\d+) months', text)
    result['term_extracted'] = int(term_match.group(1)) if term_match else None

    # Extract payment
    payment_match = re.search(r'monthly payment \$?(\d+)', text)
    result['monthly_payment_extracted'] = int(payment_match.group(1)) if payment_match else None

    # Extract penalty
    penalty_match = re.search(r'(Late fee \$\d+|Early termination fee \$\d+|None)', text)
    result['penalty_extracted'] = penalty_match.group(1) if penalty_match else None

    return result

parsed = df['raw_text'].apply(parse_contract)
parsed_df = pd.DataFrame(parsed.tolist())
parsed_df.head()

,apr_extracted,term_extracted,monthly_payment_extracted,penalty_extracted
0,10.49,24,959,None
1,3.78,24,804,Early termination fee $300
2,5.41,24,774,Late fee $50
3,5.26,48,1028,Late fee $25
4,5.97,36,1180,None


In [22]:
def risk_rule(row):
    if row['apr'] > 10:
        return "HIGH"
    if row['monthly_payment'] > 900:
        return "MEDIUM"
    return "LOW"

df['rule_based_risk'] = df.apply(risk_rule, axis=1)
df[['raw_text', 'rule_based_risk']].head()

,raw_text,rule_based_risk
0,This contract between Prime Auto and Emily Sto...,HIGH
1,This contract between ABC Motors and Priya Sha...,LOW
2,This contract between DriveEasy Finance and Sa...,LOW
3,This contract between National Motors and Alex...,MEDIUM
4,This contract between National Motors and John...,MEDIUM
